# Analytical Solver Conditioning Diagnostics

This notebook visualizes where the closed-form Bradler analytical solve becomes numerically fragile.

It uses `analytical_conditioning_sweep()` to collect boundary-condition matrix diagnostics without changing the underlying analytical solution path.
This notebook is compatible with the Bradler-style axial tip offset parameter `cant.tip_offset`; when that offset is nonzero, the contact location is `cant.contact_x = L - tip_offset`.


## Imports

In [ ]:
import os, sys
# Use the package from this repository even if it has not been pip-installed.
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))
import numpy as np
import matplotlib.pyplot as plt

from eb_cr_afm import (
    default_params,
    default_params_multi75,
    analytical_conditioning_sweep,
    frequency_sweep,
    sho_analysis,
)
from eb_cr_afm.params import ContactParams

## Helper plotting function

In [ ]:
def plot_conditioning_summary(diag, sweep_result=None, title=None):
    freqs_khz = diag["freqs"] / 1e3
    success = diag["success"]

    fig, axes = plt.subplots(3, 1, figsize=(10, 11), sharex=True)

    ax = axes[0]
    ax.semilogy(freqs_khz, diag["cond_raw"], label="Raw 2x2 system", lw=2)
    ax.semilogy(freqs_khz, diag["cond_row_scaled"], label="Row-scaled", lw=2)
    ax.semilogy(freqs_khz, diag["cond_rowcol_scaled"], label="Row+column scaled", lw=2)
    ax.set_ylabel("Condition number")
    ax.grid(True, alpha=0.3)
    ax.legend()

    bad = ~success
    if np.any(bad):
        ax.scatter(freqs_khz[bad], np.full(np.sum(bad), np.nanmax(diag["cond_raw"]) if np.isfinite(diag["cond_raw"]).any() else 1.0),
                   color="crimson", marker="x", s=60, label="Analytical failure")

    ax = axes[1]
    ax.semilogy(freqs_khz, diag["residual_raw"], label="Raw residual", lw=2)
    ax.semilogy(freqs_khz, diag["residual_scaled"], label="Scaled residual", lw=2)
    ax.semilogy(freqs_khz, diag["solution_rel_diff_scaled_vs_raw"], label="Scaled vs raw coeff. diff.", lw=2)
    ax.set_ylabel("Residual / relative diff")
    ax.grid(True, alpha=0.3)
    ax.legend()

    ax = axes[2]
    ax.semilogy(freqs_khz, np.maximum(diag["det_abs"], 1e-300), label="|det(M)|", lw=2)
    ax.set_ylabel("|det(M)|")
    ax.set_xlabel("Frequency (kHz)")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")

    if sweep_result is not None:
        modes = sho_analysis(sweep_result)
        for mode in modes:
            f_res = mode["f_res"] / 1e3
            for ax in axes:
                ax.axvline(f_res, color="0.4", ls="--", lw=1, alpha=0.5)

    if title is not None:
        fig.suptitle(title)

    fig.tight_layout()
    return fig, axes

## Example 1: Bradler 2016 Table-I-style default case

In [ ]:
cant, contact, exc, opts = default_params()
opts.method = "analytical"
opts.f_start = 300e3
opts.f_stop = 400e3
opts.f_step = 500.0

diag_default = analytical_conditioning_sweep(cant, contact, exc, opts)
sweep_default = frequency_sweep(cant, contact, exc, opts)
modes_default = sho_analysis(sweep_default)

print("Detected modes:")
for i, mode in enumerate(modes_default, start=1):
    print(f"  Mode {i}: f_res = {mode['f_res']/1e3:.3f} kHz, Q = {mode['Q']:.2f}, converged = {mode['converged']}")

print()
print(f"Successful analytical points: {diag_default['success'].sum()} / {len(diag_default['success'])}")
print(f"Max raw cond: {np.nanmax(diag_default['cond_raw']):.3e}")
print(f"Max row+col scaled cond: {np.nanmax(diag_default['cond_rowcol_scaled']):.3e}")

In [ ]:
plot_conditioning_summary(
    diag_default,
    sweep_result=sweep_default,
    title="Default Bradler-style case: analytical conditioning diagnostics",
);

## Example 2: Multi75-style PFM case

This case is closer to the first-mode contact-resonance range often used in practical CR-PFM.

In [ ]:
cant_m, contact_m, exc_m, opts_m = default_params_multi75()
opts_m.method = "analytical"
opts_m.f_start = 50e3
opts_m.f_stop = 400e3
opts_m.f_step = 500.0

diag_m = analytical_conditioning_sweep(cant_m, contact_m, exc_m, opts_m)
sweep_m = frequency_sweep(cant_m, contact_m, exc_m, opts_m)
modes_m = sho_analysis(sweep_m)

print("Detected modes:")
for i, mode in enumerate(modes_m, start=1):
    print(f"  Mode {i}: f_res = {mode['f_res']/1e3:.3f} kHz, Q = {mode['Q']:.2f}, converged = {mode['converged']}")

print()
print(f"Successful analytical points: {diag_m['success'].sum()} / {len(diag_m['success'])}")
print(f"Max raw cond: {np.nanmax(diag_m['cond_raw']):.3e}")
print(f"Max row+col scaled cond: {np.nanmax(diag_m['cond_rowcol_scaled']):.3e}")

In [ ]:
plot_conditioning_summary(
    diag_m,
    sweep_result=sweep_m,
    title="Multi75-style case: analytical conditioning diagnostics",
);

## Example 3: Sweep contact stiffness and compare fragility

This makes it easier to see whether poor conditioning is isolated to a narrow frequency region or grows systematically with contact stiffness.

In [ ]:
cant_s, contact_s, exc_s, opts_s = default_params_multi75()
opts_s.method = "analytical"
opts_s.f_start = 50e3
opts_s.f_stop = 400e3
opts_s.f_step = 1000.0

k1_values = [1.0, 10.0, 30.0, 100.0, 300.0, 1000.0]
summary_rows = []

for k1 in k1_values:
    c_use = ContactParams(
        k1=k1,
        k2=0.85 * k1,
        c1=contact_s.c1,
        c2=0.85 * contact_s.c1,
        psi=contact_s.psi,
    )
    diag = analytical_conditioning_sweep(cant_s, c_use, exc_s, opts_s)
    summary_rows.append({
        "k1": k1,
        "success_frac": diag["success"].mean(),
        "max_cond_raw": np.nanmax(diag["cond_raw"]),
        "max_cond_scaled": np.nanmax(diag["cond_rowcol_scaled"]),
        "min_det_abs": np.nanmin(diag["det_abs"]),
    })

for row in summary_rows:
    print(
        f"k1 = {row['k1']:7.1f} N/m   success = {100*row['success_frac']:5.1f}%   "
        f"max cond(raw) = {row['max_cond_raw']:.3e}   "
        f"max cond(scaled) = {row['max_cond_scaled']:.3e}   "
        f"min |det(M)| = {row['min_det_abs']:.3e}"
    )

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)

axes[0].semilogy(k1_values, [row['max_cond_raw'] for row in summary_rows], marker='o', lw=2, label='Raw')
axes[0].semilogy(k1_values, [row['max_cond_scaled'] for row in summary_rows], marker='o', lw=2, label='Row+column scaled')
axes[0].set_ylabel('Worst condition number')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(k1_values, [100*row['success_frac'] for row in summary_rows], marker='o', lw=2)
axes[1].set_ylabel('Analytical success (%)')
axes[1].grid(True, alpha=0.3)

axes[2].semilogy(k1_values, [row['min_det_abs'] for row in summary_rows], marker='o', lw=2)
axes[2].set_ylabel('Minimum |det(M)|')
axes[2].set_xlabel('Contact stiffness k1 (N/m)')
axes[2].grid(True, alpha=0.3)

fig.suptitle('How analytical conditioning changes with contact stiffness')
fig.tight_layout()

## Example 4: Inspect failed frequencies directly

If a sweep contains failures, this cell lists the affected frequencies and the recorded error strings.

In [ ]:
failed_idx = np.where(~diag_m['success'])[0]
if len(failed_idx) == 0:
    print('No analytical failures in the selected Multi75-style sweep.')
else:
    for idx in failed_idx:
        print(f"{diag_m['freqs'][idx]/1e3:.3f} kHz -> {diag_m['errors'][idx]}")